<a href="https://colab.research.google.com/github/Bhuvi285/Advanced-GenAI-Internship-Innomatics-Research-Labs/blob/main/NLP_Assignment5_POS_Chunking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NLP Assignment 5 — Fine-Tuning BERT for POS Tagging & Chunking

**Internship:** Data Science Internship – February 2026 Cohort  
**Task:** Token Classification using Transformer Models  
**Model:** DistilBERT (distilbert-base-uncased)  
**Dataset:** CoNLL-2003  

---

## Pipeline Overview
```
Raw Data → Tokenization → Label Alignment → Model Training → Evaluation → Inference → Comparison
```

## Tasks
| Task | Description | Weightage |
|------|-------------|----------|
| Task 1 | Dataset Selection | 10% |
| Task 2 | Data Preprocessing | 15% |
| Task 3 | Model Setup | 15% |
| Task 4 | Training | 20% |
| Task 5 | Evaluation | 15% |
| Task 6 | Inference | 10% |
| Task 7 | Comparison | 10% |
| Task 8 | Report/Blog | 5% |

---
## ⚙️ Setup — Install Dependencies

> **Important:** Go to `Runtime → Change runtime type → T4 GPU` before running.

In [1]:
import os

# Disable ALL progress bars
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["DISABLE_TQDM"] = "1"

# Explicitly disable tqdm to ensure no progress bars appear
from tqdm.auto import tqdm
tqdm.disable = True

from transformers.utils import logging as transformers_logging
transformers_logging.set_verbosity_error() # Only show actual errors, hide warnings


In [2]:
!pip install -q transformers datasets accelerate evaluate seqeval

In [3]:
import numpy as np
import torch
from datasets import load_dataset, Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)
import evaluate
import requests

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")
print(f"Torch: {torch.__version__}")

import transformers
print(f"Transformers: {transformers.__version__}")

Using device: cuda
Torch: 2.10.0+cu128
Transformers: 5.0.0


---
## Task 1 — Dataset Selection (10%)

**Dataset chosen:** CoNLL-2003  
**Reason:** Widely used benchmark for NER, POS tagging, and Chunking. Directly available via HuggingFace `datasets` library.

### Label Types Present:
- **POS Tags:** Fine-grained grammatical categories (NN, VBZ, JJ, DT, etc.)
- **Chunk Tags:** Phrase-level labels in BIO format (B-NP, I-NP, B-VP, O, etc.)

In [4]:
import requests
from datasets import Dataset, DatasetDict

def parse_conll(text):
    sentences_tokens, sentences_pos, sentences_chunk, sentences_ner = [], [], [], []
    tokens, pos, chunk, ner = [], [], [], []

    for line in text.strip().split('\n'):
        if line.startswith('-DOCSTART-') or line == '':
            if tokens:
                sentences_tokens.append(tokens)
                sentences_pos.append(pos)
                sentences_chunk.append(chunk)
                sentences_ner.append(ner)
                tokens, pos, chunk, ner = [], [], [], []
        else:
            parts = line.split()
            if len(parts) == 4:
                tokens.append(parts[0])
                pos.append(parts[1])
                chunk.append(parts[2])
                ner.append(parts[3])
    return sentences_tokens, sentences_pos, sentences_chunk, sentences_ner

base = "https://raw.githubusercontent.com/patverga/torch-ner-nlp-from-scratch/master/data/conll2003/"
splits = {}
for split, fname in [("train","eng.train"), ("validation","eng.testa"), ("test","eng.testb")]:
    r = requests.get(base + fname)
    toks, pos, chunk, ner = parse_conll(r.text)
    splits[split] = Dataset.from_dict({
        "tokens": toks, "pos_tags": pos,
        "chunk_tags": chunk, "ner_tags": ner
    })

dataset = DatasetDict(splits)
print("Dataset loaded!")
print(dataset)
print("\nSample:", dataset['train'][0])

Dataset loaded!
DatasetDict({
    train: Dataset({
        features: ['tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 14040
    })
    validation: Dataset({
        features: ['tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3249
    })
    test: Dataset({
        features: ['tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3452
    })
})

Sample: {'tokens': ['EU', 'rejects', 'German', 'call', 'to', 'boycott', 'British', 'lamb', '.'], 'pos_tags': ['NNP', 'VBZ', 'JJ', 'NN', 'TO', 'VB', 'JJ', 'NN', '.'], 'chunk_tags': ['I-NP', 'I-VP', 'I-NP', 'I-NP', 'I-VP', 'I-VP', 'I-NP', 'I-NP', 'O'], 'ner_tags': ['I-ORG', 'O', 'I-MISC', 'O', 'O', 'O', 'I-MISC', 'O', 'O']}


In [5]:
# Inspect the dataset structure
print("=== Dataset Features ===")
print(dataset['train'].features)
print()

# View a single example
example = dataset['train'][0]
print("=== Sample Example ===")
print(f"Tokens  : {example['tokens']}")
print(f"POS tags: {example['pos_tags']}")
print(f"Chunk   : {example['chunk_tags']}")
print(f"NER     : {example['ner_tags']}")

=== Dataset Features ===
{'tokens': List(Value('string')), 'pos_tags': List(Value('string')), 'chunk_tags': List(Value('string')), 'ner_tags': List(Value('string'))}

=== Sample Example ===
Tokens  : ['EU', 'rejects', 'German', 'call', 'to', 'boycott', 'British', 'lamb', '.']
POS tags: ['NNP', 'VBZ', 'JJ', 'NN', 'TO', 'VB', 'JJ', 'NN', '.']
Chunk   : ['I-NP', 'I-VP', 'I-NP', 'I-NP', 'I-VP', 'I-VP', 'I-NP', 'I-NP', 'O']
NER     : ['I-ORG', 'O', 'I-MISC', 'O', 'O', 'O', 'I-MISC', 'O', 'O']


In [6]:
pos_label_list = sorted(set(tag for sent in dataset['train']['pos_tags'] for tag in sent))
chunk_label_list = sorted(set(tag for sent in dataset['train']['chunk_tags'] for tag in sent))

# Create label → id mappings (since tags are strings now, not integers)
pos_label2id = {l: i for i, l in enumerate(pos_label_list)}
pos_id2label = {i: l for i, l in enumerate(pos_label_list)}
chunk_label2id = {l: i for i, l in enumerate(chunk_label_list)}
chunk_id2label = {i: l for i, l in enumerate(chunk_label_list)}

print(f"POS labels ({len(pos_label_list)}): {pos_label_list}")
print(f"Chunk labels ({len(chunk_label_list)}): {chunk_label_list}")

POS labels (45): ['"', '$', "''", '(', ')', ',', '.', ':', 'CC', 'CD', 'DT', 'EX', 'FW', 'IN', 'JJ', 'JJR', 'JJS', 'LS', 'MD', 'NN', 'NNP', 'NNPS', 'NNS', 'NN|SYM', 'PDT', 'POS', 'PRP', 'PRP$', 'RB', 'RBR', 'RBS', 'RP', 'SYM', 'TO', 'UH', 'VB', 'VBD', 'VBG', 'VBN', 'VBP', 'VBZ', 'WDT', 'WP', 'WP$', 'WRB']
Chunk labels (17): ['B-ADJP', 'B-ADVP', 'B-NP', 'B-PP', 'B-SBAR', 'B-VP', 'I-ADJP', 'I-ADVP', 'I-CONJP', 'I-INTJ', 'I-LST', 'I-NP', 'I-PP', 'I-PRT', 'I-SBAR', 'I-VP', 'O']


In [7]:
example = dataset['train'][0]
print(f"{'Word':<15} {'POS Tag':<12} {'Chunk Tag'}")
print("-" * 40)
for tok, pos, chunk in zip(example['tokens'], example['pos_tags'], example['chunk_tags']):
    print(f"{tok:<15} {pos:<12} {chunk}")

Word            POS Tag      Chunk Tag
----------------------------------------
EU              NNP          I-NP
rejects         VBZ          I-VP
German          JJ           I-NP
call            NN           I-NP
to              TO           I-VP
boycott         VB           I-VP
British         JJ           I-NP
lamb            NN           I-NP
.               .            O


**Observation:** CoNLL-2003 uses BIO (Beginning-Inside-Outside) format for chunk tags. For example:
- `B-NP` = Beginning of a Noun Phrase
- `I-NP` = Inside (continuation) of a Noun Phrase
- `O`    = Outside any chunk (independent word)

---
## Task 2 — Data Preprocessing (15%)

### Key Challenge: Subword Tokenization
BERT uses WordPiece tokenization which splits words into subwords.  
Example: `"playing"` → `["play", "##ing"]`

But labels are assigned **per word**, not per subword.  
**Solution:** Keep the label only on the **first subword** of each word, and assign `-100` to all subsequent subword tokens.  
(`-100` is PyTorch's ignore index — the loss function skips these positions.)

### Expected Outputs per example:
- `input_ids` — tokenized integer IDs
- `attention_mask` — 1 for real tokens, 0 for padding
- `labels` — aligned label IDs (-100 for subwords and special tokens)

In [8]:
# ---- Configuration ----
# We will train TWO separate models: one for POS tagging, one for Chunking
# For this notebook we demonstrate BOTH using the same preprocessing pipeline

MODEL_CHECKPOINT = "distilbert-base-uncased"  # Lightweight BERT variant
MAX_LENGTH = 128

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)
print(f"Tokenizer loaded: {MODEL_CHECKPOINT}")

# Quick subword demonstration
sample_word = "playing"
tokens = tokenizer.tokenize(sample_word)
print(f"\nSubword demo: '{sample_word}' → {tokens}")
print("→ Only the first subword gets the real label; rest get -100")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Tokenizer loaded: distilbert-base-uncased

Subword demo: 'playing' → ['playing']
→ Only the first subword gets the real label; rest get -100


In [9]:
# ---- Label Alignment Function ----
# This is the most critical part of the preprocessing

def tokenize_and_align_labels(examples, label_column):
    """
    Tokenizes input words and aligns the word-level labels
    to the subword-level tokens produced by BERT tokenizer.

    Args:
        examples: a batch of dataset examples
        label_column: 'pos_tags' or 'chunk_tags'

    Returns:
        tokenized_inputs with aligned 'labels' field
    """
    # Tokenize the words (is_split_into_words=True because input is already split)
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        max_length=MAX_LENGTH,
        is_split_into_words=True  # Input is a list of words, not a single string
    )

    all_labels = []

    for i, label_ids in enumerate(examples[label_column]):
        # word_ids() maps each subword token → its original word index
        # Returns None for special tokens ([CLS], [SEP])
        word_ids = tokenized_inputs.word_ids(batch_index=i)

        aligned_labels = []
        previous_word_id = None

        for word_id in word_ids:
            if word_id is None:
                # Special token ([CLS] or [SEP]) → ignore in loss
                aligned_labels.append(-100)
            elif word_id != previous_word_id:
                # First subword of a new word → assign real label
                aligned_labels.append(label_ids[word_id])
            else:
                # Subsequent subword of same word → ignore in loss
                aligned_labels.append(-100)

            previous_word_id = word_id

        all_labels.append(aligned_labels)

    tokenized_inputs["labels"] = all_labels
    return tokenized_inputs

print("Label alignment function defined.")

Label alignment function defined.


In [10]:
def tokenize_and_align_labels(examples, label_column, label2id):
    """
    Tokenizes input words and aligns word-level string labels
    to subword-level tokens. Converts string labels to integer IDs.
    """
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        max_length=MAX_LENGTH,
        is_split_into_words=True
    )

    all_labels = []

    for i, label_strs in enumerate(examples[label_column]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        aligned_labels = []
        previous_word_id = None

        for word_id in word_ids:
            if word_id is None:
                aligned_labels.append(-100)
            elif word_id != previous_word_id:
                # Convert string label → integer ID
                aligned_labels.append(label2id[label_strs[word_id]])
            else:
                aligned_labels.append(-100)
            previous_word_id = word_id

        all_labels.append(aligned_labels)

    tokenized_inputs["labels"] = all_labels
    return tokenized_inputs

print("Label alignment function defined.")

Label alignment function defined.


In [11]:
from datasets import disable_progress_bars
disable_progress_bars()

print("Preprocessing POS Tagging dataset...")
pos_tokenized_dataset = dataset.map(
    lambda x: tokenize_and_align_labels(x, 'pos_tags', pos_label2id),
    batched=True,
    remove_columns=dataset['train'].column_names
)

print("Preprocessing Chunking dataset...")
chunk_tokenized_dataset = dataset.map(
    lambda x: tokenize_and_align_labels(x, 'chunk_tags', chunk_label2id),
    batched=True,
    remove_columns=dataset['train'].column_names
)

print("Both datasets ready!")
print(pos_tokenized_dataset)

Preprocessing POS Tagging dataset...
Preprocessing Chunking dataset...
Both datasets ready!
DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 14040
    })
    validation: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 3249
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 3452
    })
})


---
## Task 3 — Model Setup (15%)

**Model:** `distilbert-base-uncased`  
**Class:** `AutoModelForTokenClassification`

Requirements:
- `num_labels` must match the number of unique label types in the dataset
- `id2label` — maps integer IDs → human-readable label strings
- `label2id` — maps label strings → integer IDs

We set up **two separate models** — one for POS tagging, one for chunking.

In [12]:
# ---- Build label mappings ----

# POS label mappings
pos_id2label = {i: label for i, label in enumerate(pos_label_list)}
pos_label2id = {label: i for i, label in enumerate(pos_label_list)}

# Chunk label mappings
chunk_id2label = {i: label for i, label in enumerate(chunk_label_list)}
chunk_label2id = {label: i for i, label in enumerate(chunk_label_list)}

print(f"POS  → num_labels: {len(pos_label_list)}")
print(f"id2label sample: {dict(list(pos_id2label.items())[:5])}")
print()
print(f"Chunk → num_labels: {len(chunk_label_list)}")
print(f"id2label sample: {dict(list(chunk_id2label.items())[:5])}")

POS  → num_labels: 45
id2label sample: {0: '"', 1: '$', 2: "''", 3: '(', 4: ')'}

Chunk → num_labels: 17
id2label sample: {0: 'B-ADJP', 1: 'B-ADVP', 2: 'B-NP', 3: 'B-PP', 4: 'B-SBAR'}


In [13]:
# ---- Load POS Tagging Model ----
pos_model = AutoModelForTokenClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=len(pos_label_list),
    id2label=pos_id2label,
    label2id=pos_label2id
)
print(f"POS Model loaded | Output labels: {pos_model.config.num_labels}")

POS Model loaded | Output labels: 45


In [14]:
# ---- Load Chunking Model ----
chunk_model = AutoModelForTokenClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=len(chunk_label_list),
    id2label=chunk_id2label,
    label2id=chunk_label2id
)
print(f"Chunk Model loaded | Output labels: {chunk_model.config.num_labels}")

Chunk Model loaded | Output labels: 17


In [15]:
# ---- Data collator ----
# Handles dynamic padding within each batch (pads to longest sequence in batch)
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)
print("Data collator ready (dynamic padding enabled)")

Data collator ready (dynamic padding enabled)


---
## Task 4 — Training (20%)

**Trainer API:** HuggingFace `Trainer` abstracts the training loop.  
We define `TrainingArguments` with:
- Learning rate: `2e-5` (standard for BERT fine-tuning)
- Epochs: `3`
- Batch size: `16`
- Evaluation strategy: after each epoch

In [16]:
# ---- seqeval metric setup ----
# seqeval evaluates sequence-level predictions (entity spans), not token accuracy
seqeval = evaluate.load("seqeval")

def build_compute_metrics(label_list):
    """
    Returns a compute_metrics function for the given label list.
    seqeval expects lists of label strings (not IDs), so we convert here.
    """
    def compute_metrics(eval_preds):
        logits, labels = eval_preds

        # Get the predicted class for each token
        predictions = np.argmax(logits, axis=-1)

        # Remove -100 positions and convert IDs to label strings
        true_labels = [
            [label_list[l] for l in label_seq if l != -100]
            for label_seq in labels
        ]
        true_predictions = [
            [label_list[p] for p, l in zip(pred_seq, label_seq) if l != -100]
            for pred_seq, label_seq in zip(predictions, labels)
        ]

        results = seqeval.compute(
            predictions=true_predictions,
            references=true_labels
        )
        return {
            "precision": results["overall_precision"],
            "recall":    results["overall_recall"],
            "f1":        results["overall_f1"],
            "accuracy":  results["overall_accuracy"]
        }
    return compute_metrics

print("compute_metrics builder defined.")

compute_metrics builder defined.


In [17]:
# ---- Training Arguments ----
# Same hyperparameters used for both models

def build_training_args(output_dir):
    return TrainingArguments(
        output_dir=output_dir,
        learning_rate=2e-5,           # Standard BERT fine-tuning LR
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        num_train_epochs=3,
        weight_decay=0.01,            # L2 regularization
        eval_strategy="epoch",        # Evaluate after every epoch
        save_strategy="epoch",
        load_best_model_at_end=True,
        logging_steps=100,
        push_to_hub=False
    )

print("Training arguments builder defined.")

Training arguments builder defined.


In [18]:
print("=" * 50)
print("Training POS Tagging Model...")
print("=" * 50)

pos_trainer = Trainer(
    model=pos_model,
    args=build_training_args("./pos-model"),
    train_dataset=pos_tokenized_dataset['train'],
    eval_dataset=pos_tokenized_dataset['validation'],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=build_compute_metrics(pos_label_list)
)

pos_train_result = pos_trainer.train()
print("\nPOS Tagging Training Complete!")
print(pos_train_result)

Training POS Tagging Model...
{'loss': '2.076', 'grad_norm': '2.119', 'learning_rate': '1.925e-05', 'epoch': '0.1139'}
{'loss': '0.6402', 'grad_norm': '2.57', 'learning_rate': '1.849e-05', 'epoch': '0.2278'}
{'loss': '0.4054', 'grad_norm': '2.162', 'learning_rate': '1.773e-05', 'epoch': '0.3417'}
{'loss': '0.341', 'grad_norm': '1.877', 'learning_rate': '1.697e-05', 'epoch': '0.4556'}
{'loss': '0.3012', 'grad_norm': '2.269', 'learning_rate': '1.621e-05', 'epoch': '0.5695'}
{'loss': '0.2806', 'grad_norm': '2.308', 'learning_rate': '1.545e-05', 'epoch': '0.6834'}
{'loss': '0.2733', 'grad_norm': '1.717', 'learning_rate': '1.469e-05', 'epoch': '0.7973'}
{'loss': '0.2523', 'grad_norm': '1.78', 'learning_rate': '1.393e-05', 'epoch': '0.9112'}


/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: NNP seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: : seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: IN seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: NN seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: . seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarni

{'eval_loss': '0.2607', 'eval_precision': '0.91', 'eval_recall': '0.9067', 'eval_f1': '0.9083', 'eval_accuracy': '0.9365', 'eval_runtime': '7.025', 'eval_samples_per_second': '462.5', 'eval_steps_per_second': '29.04', 'epoch': '1'}
{'loss': '0.2463', 'grad_norm': '1.439', 'learning_rate': '1.317e-05', 'epoch': '1.025'}
{'loss': '0.2272', 'grad_norm': '1.668', 'learning_rate': '1.241e-05', 'epoch': '1.139'}
{'loss': '0.2142', 'grad_norm': '1.894', 'learning_rate': '1.166e-05', 'epoch': '1.253'}
{'loss': '0.2124', 'grad_norm': '1.816', 'learning_rate': '1.09e-05', 'epoch': '1.367'}
{'loss': '0.1983', 'grad_norm': '1.841', 'learning_rate': '1.014e-05', 'epoch': '1.481'}
{'loss': '0.2043', 'grad_norm': '1.853', 'learning_rate': '9.377e-06', 'epoch': '1.595'}
{'loss': '0.1945', 'grad_norm': '1.626', 'learning_rate': '8.618e-06', 'epoch': '1.708'}
{'loss': '0.1886', 'grad_norm': '2.027', 'learning_rate': '7.859e-06', 'epoch': '1.822'}
{'loss': '0.1796', 'grad_norm': '2.304', 'learning_rate':

/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: NNP seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: : seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: IN seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: NN seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: . seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarni

{'eval_loss': '0.2271', 'eval_precision': '0.9177', 'eval_recall': '0.9159', 'eval_f1': '0.9168', 'eval_accuracy': '0.9426', 'eval_runtime': '6.822', 'eval_samples_per_second': '476.3', 'eval_steps_per_second': '29.9', 'epoch': '2'}
{'loss': '0.1883', 'grad_norm': '2.084', 'learning_rate': '6.34e-06', 'epoch': '2.05'}
{'loss': '0.1668', 'grad_norm': '1.429', 'learning_rate': '5.581e-06', 'epoch': '2.164'}
{'loss': '0.1665', 'grad_norm': '2.128', 'learning_rate': '4.822e-06', 'epoch': '2.278'}
{'loss': '0.1622', 'grad_norm': '1.977', 'learning_rate': '4.062e-06', 'epoch': '2.392'}
{'loss': '0.1692', 'grad_norm': '1.796', 'learning_rate': '3.303e-06', 'epoch': '2.506'}
{'loss': '0.1786', 'grad_norm': '1.803', 'learning_rate': '2.544e-06', 'epoch': '2.62'}
{'loss': '0.162', 'grad_norm': '1.939', 'learning_rate': '1.784e-06', 'epoch': '2.733'}
{'loss': '0.1622', 'grad_norm': '2.546', 'learning_rate': '1.025e-06', 'epoch': '2.847'}
{'loss': '0.1536', 'grad_norm': '2.591', 'learning_rate': '

/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: NNP seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: : seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: IN seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: NN seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: . seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarni

{'eval_loss': '0.2222', 'eval_precision': '0.9212', 'eval_recall': '0.9203', 'eval_f1': '0.9208', 'eval_accuracy': '0.9453', 'eval_runtime': '6.125', 'eval_samples_per_second': '530.4', 'eval_steps_per_second': '33.3', 'epoch': '3'}
{'train_runtime': '270.1', 'train_samples_per_second': '155.9', 'train_steps_per_second': '9.751', 'train_loss': '0.3036', 'epoch': '3'}

POS Tagging Training Complete!
TrainOutput(global_step=2634, training_loss=0.30363637819920153, metrics={'train_runtime': 270.1324, 'train_samples_per_second': 155.924, 'train_steps_per_second': 9.751, 'train_loss': 0.30363637819920153, 'epoch': 3.0})


In [19]:
print("=" * 50)
print("Training Chunking Model...")
print("=" * 50)

chunk_trainer = Trainer(
    model=chunk_model,
    args=build_training_args("./chunk-model"),
    train_dataset=chunk_tokenized_dataset['train'],
    eval_dataset=chunk_tokenized_dataset['validation'],
    # tokenizer=tokenizer, # Remove this line
    data_collator=data_collator,
    compute_metrics=build_compute_metrics(chunk_label_list)
)

chunk_train_result = chunk_trainer.train()
print("\nChunking Training Complete!")
print(chunk_train_result)

Training Chunking Model...
{'loss': '0.8791', 'grad_norm': '1.822', 'learning_rate': '1.925e-05', 'epoch': '0.1139'}
{'loss': '0.2721', 'grad_norm': '1.583', 'learning_rate': '1.849e-05', 'epoch': '0.2278'}
{'loss': '0.2164', 'grad_norm': '1.171', 'learning_rate': '1.773e-05', 'epoch': '0.3417'}
{'loss': '0.1941', 'grad_norm': '1.325', 'learning_rate': '1.697e-05', 'epoch': '0.4556'}
{'loss': '0.1826', 'grad_norm': '1.074', 'learning_rate': '1.621e-05', 'epoch': '0.5695'}
{'loss': '0.1715', 'grad_norm': '1.342', 'learning_rate': '1.545e-05', 'epoch': '0.6834'}
{'loss': '0.1579', 'grad_norm': '1.058', 'learning_rate': '1.469e-05', 'epoch': '0.7973'}
{'loss': '0.1561', 'grad_norm': '1.237', 'learning_rate': '1.393e-05', 'epoch': '0.9112'}


/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


{'eval_loss': '0.1594', 'eval_precision': '0.909', 'eval_recall': '0.9024', 'eval_f1': '0.9057', 'eval_accuracy': '0.9608', 'eval_runtime': '5.356', 'eval_samples_per_second': '606.6', 'eval_steps_per_second': '38.09', 'epoch': '1'}
{'loss': '0.1482', 'grad_norm': '1.158', 'learning_rate': '1.317e-05', 'epoch': '1.025'}
{'loss': '0.1442', 'grad_norm': '1.139', 'learning_rate': '1.241e-05', 'epoch': '1.139'}
{'loss': '0.1292', 'grad_norm': '1.761', 'learning_rate': '1.166e-05', 'epoch': '1.253'}
{'loss': '0.1299', 'grad_norm': '0.7932', 'learning_rate': '1.09e-05', 'epoch': '1.367'}
{'loss': '0.1349', 'grad_norm': '1.066', 'learning_rate': '1.014e-05', 'epoch': '1.481'}
{'loss': '0.1255', 'grad_norm': '1.26', 'learning_rate': '9.377e-06', 'epoch': '1.595'}
{'loss': '0.1194', 'grad_norm': '1.599', 'learning_rate': '8.618e-06', 'epoch': '1.708'}
{'loss': '0.1186', 'grad_norm': '1.242', 'learning_rate': '7.859e-06', 'epoch': '1.822'}
{'loss': '0.1164', 'grad_norm': '1.788', 'learning_rate'

/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


{'eval_loss': '0.1382', 'eval_precision': '0.919', 'eval_recall': '0.9073', 'eval_f1': '0.9131', 'eval_accuracy': '0.9638', 'eval_runtime': '5.466', 'eval_samples_per_second': '594.4', 'eval_steps_per_second': '37.32', 'epoch': '2'}
{'loss': '0.1249', 'grad_norm': '2.031', 'learning_rate': '6.34e-06', 'epoch': '2.05'}
{'loss': '0.1064', 'grad_norm': '0.8281', 'learning_rate': '5.581e-06', 'epoch': '2.164'}
{'loss': '0.09933', 'grad_norm': '1.44', 'learning_rate': '4.822e-06', 'epoch': '2.278'}
{'loss': '0.09889', 'grad_norm': '1.244', 'learning_rate': '4.062e-06', 'epoch': '2.392'}
{'loss': '0.09988', 'grad_norm': '2.229', 'learning_rate': '3.303e-06', 'epoch': '2.506'}
{'loss': '0.1123', 'grad_norm': '1.555', 'learning_rate': '2.544e-06', 'epoch': '2.62'}
{'loss': '0.1064', 'grad_norm': '1.304', 'learning_rate': '1.784e-06', 'epoch': '2.733'}
{'loss': '0.09952', 'grad_norm': '1.176', 'learning_rate': '1.025e-06', 'epoch': '2.847'}
{'loss': '0.1039', 'grad_norm': '1.965', 'learning_rat

/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


{'eval_loss': '0.1367', 'eval_precision': '0.9208', 'eval_recall': '0.9101', 'eval_f1': '0.9154', 'eval_accuracy': '0.965', 'eval_runtime': '5.465', 'eval_samples_per_second': '594.5', 'eval_steps_per_second': '37.33', 'epoch': '3'}
{'train_runtime': '353.1', 'train_samples_per_second': '119.3', 'train_steps_per_second': '7.46', 'train_loss': '0.1663', 'epoch': '3'}

Chunking Training Complete!
TrainOutput(global_step=2634, training_loss=0.1662912364248625, metrics={'train_runtime': 353.0922, 'train_samples_per_second': 119.289, 'train_steps_per_second': 7.46, 'train_loss': 0.1662912364248625, 'epoch': 3.0})


---
## Task 5 — Evaluation (15%)

Using `seqeval` metric which computes:
- **Precision** — Of all predicted tags, how many were correct?
- **Recall** — Of all actual tags, how many did the model find?
- **F1 Score** — Harmonic mean of Precision and Recall

In [20]:
# ---- Evaluate POS Tagging Model on Test Set ----
print("=== POS Tagging Evaluation (Test Set) ===")
pos_results = pos_trainer.evaluate(eval_dataset=pos_tokenized_dataset['test'])

print(f"Precision : {pos_results['eval_precision']:.4f}")
print(f"Recall    : {pos_results['eval_recall']:.4f}")
print(f"F1 Score  : {pos_results['eval_f1']:.4f}")
print(f"Accuracy  : {pos_results['eval_accuracy']:.4f}")

=== POS Tagging Evaluation (Test Set) ===


/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: NN seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: : seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: NNP seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: VB seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: , seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarni

{'eval_loss': '0.2652', 'eval_precision': '0.9118', 'eval_recall': '0.9084', 'eval_f1': '0.9101', 'eval_accuracy': '0.9391', 'eval_runtime': '5.7', 'eval_samples_per_second': '605.6', 'eval_steps_per_second': '37.89', 'epoch': '3'}
Precision : 0.9118
Recall    : 0.9084
F1 Score  : 0.9101
Accuracy  : 0.9391


In [21]:
# ---- Evaluate Chunking Model on Test Set ----
print("=== Chunking Evaluation (Test Set) ===")
chunk_results = chunk_trainer.evaluate(eval_dataset=chunk_tokenized_dataset['test'])

print(f"Precision : {chunk_results['eval_precision']:.4f}")
print(f"Recall    : {chunk_results['eval_recall']:.4f}")
print(f"F1 Score  : {chunk_results['eval_f1']:.4f}")
print(f"Accuracy  : {chunk_results['eval_accuracy']:.4f}")

=== Chunking Evaluation (Test Set) ===
{'eval_loss': '0.1541', 'eval_precision': '0.9124', 'eval_recall': '0.8974', 'eval_f1': '0.9048', 'eval_accuracy': '0.9625', 'eval_runtime': '4.867', 'eval_samples_per_second': '709.3', 'eval_steps_per_second': '44.38', 'epoch': '3'}
Precision : 0.9124
Recall    : 0.8974
F1 Score  : 0.9048
Accuracy  : 0.9625


In [22]:
# ---- Summary Comparison Table ----
print("=" * 55)
print(f"{'Metric':<12} {'POS Tagging':>18} {'Chunking':>18}")
print("=" * 55)
print(f"{'Precision':<12} {pos_results['eval_precision']:>18.4f} {chunk_results['eval_precision']:>18.4f}")
print(f"{'Recall':<12} {pos_results['eval_recall']:>18.4f} {chunk_results['eval_recall']:>18.4f}")
print(f"{'F1 Score':<12} {pos_results['eval_f1']:>18.4f} {chunk_results['eval_f1']:>18.4f}")
print(f"{'Accuracy':<12} {pos_results['eval_accuracy']:>18.4f} {chunk_results['eval_accuracy']:>18.4f}")
print("=" * 55)

Metric              POS Tagging           Chunking
Precision                0.9118             0.9124
Recall                   0.9084             0.8974
F1 Score                 0.9101             0.9048
Accuracy                 0.9391             0.9625


---
## Task 6 — Inference (10%)

Running the trained models on a custom sentence to predict POS tags and chunk tags.

**Example sentence:** `"John works at Google in California"`

In [23]:
# ---- Inference Helper Function ----

def predict_tags(sentence, model, tokenizer, label_list):
    """
    Predict token-level tags for a given sentence.

    Args:
        sentence: input string
        model: fine-tuned token classification model
        tokenizer: BERT tokenizer
        label_list: list of label strings

    Returns:
        List of (word, predicted_label) tuples
    """
    model.eval()
    words = sentence.split()

    # Tokenize
    inputs = tokenizer(
        words,
        is_split_into_words=True,
        return_tensors="pt",
        truncation=True,
        max_length=128
    ).to(device)

    model.to(device)

    with torch.no_grad():
        outputs = model(**inputs)

    # Get predicted class per token
    predictions = torch.argmax(outputs.logits, dim=-1)[0].cpu().numpy()

    # Map predictions back to words (skip subwords and special tokens)
    word_ids = inputs.word_ids(batch_index=0)
    results = []
    seen_words = set()

    for token_idx, word_id in enumerate(word_ids):
        if word_id is None:
            continue  # Skip [CLS] and [SEP]
        if word_id in seen_words:
            continue  # Skip subword continuations
        seen_words.add(word_id)
        predicted_label = label_list[predictions[token_idx]]
        results.append((words[word_id], predicted_label))

    return results

print("Inference function defined.")

Inference function defined.


In [24]:
# ---- Run inference on example sentence ----
test_sentence = "John works at Google in California"
print(f"Input sentence: '{test_sentence}'")
print()

# POS Tags
pos_predictions = predict_tags(test_sentence, pos_model, tokenizer, pos_label_list)
print("--- POS Tags ---")
print(f"{'Word':<15} {'POS Tag'}")
print("-" * 25)
for word, tag in pos_predictions:
    print(f"{word:<15} {tag}")

print()

# Chunk Tags
chunk_predictions = predict_tags(test_sentence, chunk_model, tokenizer, chunk_label_list)
print("--- Chunk Tags ---")
print(f"{'Word':<15} {'Chunk Tag'}")
print("-" * 25)
for word, tag in chunk_predictions:
    print(f"{word:<15} {tag}")

Input sentence: 'John works at Google in California'

--- POS Tags ---
Word            POS Tag
-------------------------
John            NNP
works           VBZ
at              IN
Google          NNP
in              IN
California      NNP

--- Chunk Tags ---
Word            Chunk Tag
-------------------------
John            I-NP
works           I-NP
at              I-PP
Google          I-NP
in              I-PP
California      I-NP


In [25]:
# ---- Full output table (both tasks together) ----
print("=" * 45)
print(f"{'Word':<15} {'POS Tag':<12} {'Chunk Tag'}")
print("=" * 45)
for (word, pos), (_, chunk) in zip(pos_predictions, chunk_predictions):
    print(f"{word:<15} {pos:<12} {chunk}")
print("=" * 45)

Word            POS Tag      Chunk Tag
John            NNP          I-NP
works           VBZ          I-NP
at              IN           I-PP
Google          NNP          I-NP
in              IN           I-PP
California      NNP          I-NP


In [26]:
# ---- Try more custom sentences ----
custom_sentences = [
    "The quick brown fox jumps over the lazy dog",
    "She sells seashells by the seashore",
    "Anthropic developed the Claude language model"
]

for sent in custom_sentences:
    print(f"\nSentence: '{sent}'")
    pos_preds = predict_tags(sent, pos_model, tokenizer, pos_label_list)
    chunk_preds = predict_tags(sent, chunk_model, tokenizer, chunk_label_list)
    print(f"{'Word':<20} {'POS':<10} {'Chunk'}")
    print("-" * 40)
    for (word, pos), (_, chunk) in zip(pos_preds, chunk_preds):
        print(f"{word:<20} {pos:<10} {chunk}")


Sentence: 'The quick brown fox jumps over the lazy dog'
Word                 POS        Chunk
----------------------------------------
The                  DT         I-NP
quick                JJ         I-NP
brown                NNP        I-NP
fox                  NNP        I-NP
jumps                VBZ        I-VP
over                 IN         I-PP
the                  DT         I-NP
lazy                 JJ         I-NP
dog                  NN         I-NP

Sentence: 'She sells seashells by the seashore'
Word                 POS        Chunk
----------------------------------------
She                  PRP        I-NP
sells                VBZ        I-VP
seashells            NNS        I-NP
by                   IN         I-PP
the                  DT         I-NP
seashore             NN         I-NP

Sentence: 'Anthropic developed the Claude language model'
Word                 POS        Chunk
----------------------------------------
Anthropic            JJ         I-NP
develo

---
## Task 7 — Comparison: POS Tagging vs Chunking (10%)

### What is POS Tagging?
**Part-of-Speech (POS) Tagging** assigns a grammatical category to every single token in a sentence.

| Word | POS Tag | Meaning |
|------|---------|--------|
| John | NNP | Proper Noun |
| works | VBZ | Verb, 3rd person singular |
| at | IN | Preposition |
| Google | NNP | Proper Noun |

- **Level:** Word/Token level
- **Difficulty:** Easier — each word gets one label independently
- **Output format:** Single flat label per token

---

### What is Chunking?
**Chunking** (Phrase Detection) groups tokens into multi-word phrases.

| Word | Chunk Tag | Phrase |
|------|-----------|-------|
| John | B-NP | Start of Noun Phrase |
| works | B-VP | Start of Verb Phrase |
| at | O | Outside any phrase |
| Google | B-NP | Start of new Noun Phrase |
| in | O | Outside |
| California | B-NP | Start of Noun Phrase |

- **Level:** Phrase/Span level  
- **Difficulty:** Harder — requires understanding context across multiple tokens  
- **Output format:** BIO (Beginning-Inside-Outside) tags

---

### Side-by-side Comparison

| Aspect | POS Tagging | Chunking |
|--------|-------------|----------|
| Granularity | Token-level | Phrase-level |
| Label format | Single tag | BIO format |
| Task difficulty | Easier | Harder |
| Output | Grammar category per word | Phrase groupings |
| Use case | Syntax analysis, NER preprocessing | IE, parsing, QA systems |
| Expected F1 | Higher (~95%+) | Lower (~92%+) |
| Context needed | Minimal | Span-level context required |

In [27]:
# ---- Programmatic comparison of results ----

print("=== Comparison: POS Tagging vs Chunking ===")
print()
print(f"{'Aspect':<25} {'POS Tagging':<20} {'Chunking'}")
print("-" * 65)
print(f"{'Granularity':<25} {'Token-level':<20} {'Phrase-level'}")
print(f"{'Label Format':<25} {'Single flat tag':<20} {'BIO (B-/I-/O)'}")
print(f"{'# of Labels':<25} {len(pos_label_list):<20} {len(chunk_label_list)}")
print(f"{'Task Difficulty':<25} {'Easier':<20} {'Medium-Hard'}")
print(f"{'F1 Score (test)':<25} {pos_results['eval_f1']:<20.4f} {chunk_results['eval_f1']:.4f}")
print(f"{'Precision (test)':<25} {pos_results['eval_precision']:<20.4f} {chunk_results['eval_precision']:.4f}")
print(f"{'Recall (test)':<25} {pos_results['eval_recall']:<20.4f} {chunk_results['eval_recall']:.4f}")

=== Comparison: POS Tagging vs Chunking ===

Aspect                    POS Tagging          Chunking
-----------------------------------------------------------------
Granularity               Token-level          Phrase-level
Label Format              Single flat tag      BIO (B-/I-/O)
# of Labels               45                   17
Task Difficulty           Easier               Medium-Hard
F1 Score (test)           0.9101               0.9048
Precision (test)          0.9118               0.9124
Recall (test)             0.9084               0.8974


---
## Task 8 — Report / Blog (5%)

### Differences Between POS Tagging and Chunking

**POS Tagging** is a token-level classification task where each word in a sentence is labeled with its grammatical role — noun, verb, adjective, preposition, etc. It is relatively straightforward because each token's tag can often be determined from local context, and the label space is flat (one tag per token).

**Chunking**, on the other hand, operates at the phrase level. It groups consecutive tokens into named syntactic units (Noun Phrases, Verb Phrases, Prepositional Phrases) using BIO notation. This requires the model to understand multi-token spans and transitions between phrase boundaries, making it a harder task with more nuanced label dependencies.

---

### Challenges Faced

1. **Subword tokenization alignment:** BERT's WordPiece tokenizer splits words into subwords, but labels exist at the word level. Careful alignment using `word_ids()` and assigning `-100` to subword continuations was essential to avoid label mismatch during training.

2. **seqeval expectations:** Unlike accuracy (token-level), `seqeval` expects sequence-level predictions as label strings, not integer IDs. The `compute_metrics` function had to handle filtering out `-100` positions and converting IDs back to label strings before evaluation.

3. **Training two separate models:** Since POS tagging and chunking have different label sets and semantics, training two independent models (rather than a multi-task model) provided cleaner results and easier debugging.

4. **Inference word-to-subword mapping:** During inference, care was needed to map predictions back from subword tokens to original words — only taking the prediction from the first subword of each word.

---

### Observations and Insights

- **POS Tagging** achieved higher F1 scores than Chunking as expected — word-level syntax is more local and predictable.
- **Chunking** requires understanding phrase boundaries which is a span-level reasoning task — harder for the model but equally well-supported by BERT's bidirectional attention.
- **DistilBERT** proved efficient for this task — achieving near-BERT performance at ~40% less compute, making it practical for internship-scale Colab training.
- The `seqeval` metric provides a realistic view of model quality — it penalizes partial phrase matches, making it more meaningful than raw token accuracy for NLP evaluation.

---


In [28]:
# ---- Final Summary ----
print("=" * 60)
print("NLP ASSIGNMENT 5 — FINAL SUMMARY")
print("=" * 60)
print(f"Model Used      : DistilBERT (distilbert-base-uncased)")
print(f"Dataset         : CoNLL-2003")
print(f"Tasks Completed : POS Tagging + Chunking")
print()
print(f"POS Tagging Results:")
print(f"  Precision : {pos_results['eval_precision']:.4f}")
print(f"  Recall    : {pos_results['eval_recall']:.4f}")
print(f"  F1 Score  : {pos_results['eval_f1']:.4f}")
print()
print(f"Chunking Results:")
print(f"  Precision : {chunk_results['eval_precision']:.4f}")
print(f"  Recall    : {chunk_results['eval_recall']:.4f}")
print(f"  F1 Score  : {chunk_results['eval_f1']:.4f}")
print()
print("All 8 tasks completed successfully!")
print("=" * 60)

NLP ASSIGNMENT 5 — FINAL SUMMARY
Model Used      : DistilBERT (distilbert-base-uncased)
Dataset         : CoNLL-2003
Tasks Completed : POS Tagging + Chunking

POS Tagging Results:
  Precision : 0.9118
  Recall    : 0.9084
  F1 Score  : 0.9101

Chunking Results:
  Precision : 0.9124
  Recall    : 0.8974
  F1 Score  : 0.9048

All 8 tasks completed successfully!
